# Teste de Extração: PPM Bovinos (Pesquisa Pecuária Municipal)
Extrai o efetivo de bovinos (Tabela 3939) via API SIDRA.

In [ ]:
import os
import pandas as pd
import sidrapy
import uuid
import time
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

DATA_DIR = Path("data")
BRONZE_DIR = DATA_DIR / "bronze"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

def salvar_parquet(df, dataset, ano):
    path = BRONZE_DIR / dataset / f"ano={ano}"
    path.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path / f"chunk_{uuid.uuid4().hex[:8]}.parquet")

def baixar_ppm(ano):
    try:
        df = sidrapy.get_table(table_code="3939", territorial_level="6", ibge_territorial_code="all", period=str(ano), classification="79/2670")
        if df is not None and len(df) > 1:
            return df.iloc[1:].copy()
    except Exception as e: print(f"Erro {ano}: {e}")
    return pd.DataFrame()

anos = [2021, 2022, 2023]
for ano in tqdm(anos, desc="PPM Bovinos"):
    dados = baixar_ppm(ano)
    if not dados.empty: salvar_parquet(dados, "ppm_bovinos", ano)
    time.sleep(1)
print("Fim!")